# TC5061 | Actividad: Dataset - Notebook | Individual
**Maestría en Inteligencia Artificial Aplicada - Machine Learning Operations (MLOps)**
**Autor:** Daniel Sánchez Hernández

## Objetivo
Recorrer de forma reproducible el flujo básico de un proyecto de Machine Learning: explorar y limpiar un conjunto de datos, entrenar y evaluar un modelo *baseline* y registrar el experimento en MLflow Tracking.

## Dataset
**AI4I 2020 Predictive Maintenance Dataset** (Matzka, S., 2020). Conjunto de datos sintético que simula datos de una fresadora industrial, pensado para tareas de mantenimiento predictivo.
- Fuente original: [Kaggle - stephanmatzka](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020) y [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/601).
- Licencia: Creative Commons Attribution 4.0 (CC BY 4.0).
- Problema a resolver: **clasificación binaria** - predecir si la máquina falla (`Machine failure`) a partir de las condiciones de operación.

## Nota de reproducibilidad
Este notebook debe ejecutarse desde la raíz del repositorio, donde existe la carpeta `data/raw/`. La configuración global (rutas, random seed y opciones de visualización) se define una sola vez, al inicio.

In [ ]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

## Configuración global

Todas las decisiones que aplican al notebook completo se concentran en esta celda, para no repetirlas en cada paso y para que cualquier persona sepa dónde cambiarlas:
- **Ruta de datos:** ubicación del archivo original, que se trata como solo lectura.
- **Random seed (`SEED`):** se usará en el train/test split y en el modelo, para que los resultados sean reproducibles en cualquier ejecución.
- **Opciones de visualización:** pandas muestra todas las columnas y el texto completo de las celdas; las gráficas usan un estilo uniforme.
- **Colores de las gráficas:** el *target* usa siempre el mismo par de colores, azul (sin falla) y naranja (falla). Se eligió una paleta apta para daltonismo porque el autor es daltónico: el par azul/naranja se distingue con los tipos más comunes de daltonismo, lo que facilita la lectura de las gráficas. Los colores se nombran con constantes (`BLUE`, `ORANGE`) en lugar de códigos hexadecimales sueltos, para que el código sea más legible.

In [ ]:
# Ruta del archivo original (solo lectura: nunca se modifica)
DATA_PATH = Path("data/raw/ai4i2020.csv")

# Random seed única para todo el notebook (reproducibilidad)
SEED = 42

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.width", 200)          # ancho suficiente para tablas amplias
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Estilo uniforme para todas las gráficas
# sns.set_theme(style="whitegrid")

# Colores con nombre (paleta apta para daltonismo, validada para distinguirse entre sí)
BLUE = "#2a78d6"
ORANGE = "#eb6834"

# Colores fijos para el target en todas las gráficas: 0 = sin falla, 1 = falla
TARGET_COLORS = {0: BLUE, 1: ORANGE}

## Paso 1: Carga y descripción del dataset

En este paso se importan los datos y se describe su estructura: número de registros, variables, tipos de dato y significado de cada columna.

**Schema validation:** después de cargar el archivo se verifica que contenga exactamente las columnas esperadas. Es una práctica básica de validación de datos en MLOps: si la fuente cambia (una columna renombrada, faltante o adicional), el notebook se detiene con un mensaje claro en lugar de fallar más adelante de forma confusa. En pipelines productivos este control se automatiza con herramientas como Great Expectations o DeepChecks.

In [ ]:
# Carga del dataset
df = pd.read_csv(DATA_PATH)

# Esquema esperado: columnas documentadas por el autor del dataset
EXPECTED_COLUMNS = ["UDI", "Product ID", "Type", "Air temperature [K]", "Process temperature [K]",
                    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
                    "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF",
]

# Comparación entre columnas esperadas y columnas reales
missing_columns = set(EXPECTED_COLUMNS) - set(df.columns)
extra_columns = set(df.columns) - set(EXPECTED_COLUMNS)

assert not missing_columns, f"Faltan columnas: {missing_columns}"
assert not extra_columns, f"Columnas no esperadas: {extra_columns}"
print("Schema validation correcta: las", len(EXPECTED_COLUMNS), "columnas esperadas están presentes.")

In [ ]:
# Dimensiones del dataset: (registros, variables)
n_rows, n_cols = df.shape
print(f"Registros: {n_rows}")
print(f"Variables: {n_cols}")

In [ ]:
# Primeras filas para ver el aspecto real de los datos
df.head()

In [ ]:
# Estructura: tipo de dato y valores no nulos por columna
df.info()

In [ ]:
# Cardinalidad: valores distintos por columna (ayuda a distinguir identificadores, categorías y numéricas)
df.nunique().to_frame(name="n_unique")

### Diccionario de datos
La siguiente tabla documenta el rol, la descripción y la unidad de cada variable. Las descripciones se conservan en inglés, tal como aparecen en la documentación original del dataset (Matzka, 2020; [Kaggle](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020); [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/601)). Se añade una columna con el tipo de dato que pandas asignó realmente, para contrastarlo con la documentación.

Observaciones:
- La documentación llama `UID` al identificador, pero en el archivo CSV la columna se llama `UDI`; se usa el nombre real del archivo.
- UCI clasifica a `TWF`, `HDF`, `PWF`, `OSF` y `RNF` como *target*, no como *features*: son los cinco modos de falla y `Machine failure` vale 1 cuando ocurre al menos uno. Este punto se retoma al construir el modelo *baseline* (Paso 4).

In [ ]:
# Diccionario de datos: descripciones tomadas de la documentación original (en inglés)
data_dictionary = pd.DataFrame(
    [
        ("UDI", "ID", "Unique identifier ranging from 1 to 10000", "-"),
        ("Product ID", "ID", "Consisting of a letter L, M, or H for low (50% of all products), medium (30%) and high (20%) as product quality variants and a variant-specific serial number", "-"),
        ("Type", "Feature", "Just the product type L, M or H from column 2", "-"),
        ("Air temperature [K]", "Feature", "Generated using a random walk process later normalized to a standard deviation of 2 K around 300 K", "K"),
        ("Process temperature [K]", "Feature", "Generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K", "K"),
        ("Rotational speed [rpm]", "Feature", "Calculated from a power of 2860 W, overlaid with a normally distributed noise", "rpm"),
        ("Torque [Nm]", "Feature", "Torque values are normally distributed around 40 Nm with a SD = 10 Nm and no negative values", "Nm"),
        ("Tool wear [min]", "Feature", "The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process", "min"),
        ("Machine failure", "Target", "Label that indicates whether the machine has failed in this particular datapoint for any of the failure modes", "-"),
        ("TWF", "Target", "Tool wear failure", "-"),
        ("HDF", "Target", "Heat dissipation failure", "-"),
        ("PWF", "Target", "Power failure", "-"),
        ("OSF", "Target", "Overstrain failure", "-"),
        ("RNF", "Target", "Random failures", "-"),
    ],
    columns=["Variable", "Role", "Description", "Units"],
)

# Tipo de dato real asignado por pandas, para contrastarlo con la documentación
data_dictionary["pandas dtype"] = data_dictionary["Variable"].map(df.dtypes.astype(str))
data_dictionary

## Paso 2: Análisis exploratorio de datos (EDA)

El objetivo del EDA es entender los datos antes de modelar: cómo se distribuye cada variable, qué tan balanceado está el *target* y qué *features* parecen relacionarse con las fallas. Los hallazgos de este paso orientan las decisiones de limpieza (Paso 3) y del modelo *baseline* (Paso 4).

Se analizan:
- **Estadística descriptiva** de las *features* numéricas y de la variable categórica `Type`.
- **Balance de clases** del *target* `Machine failure`.
- **Modos de falla** y su consistencia con el *target*.
- **Distribuciones** de las *features* numéricas.
- **Relaciones** entre las *features* y el *target*, y entre las propias *features* (correlación).

Los identificadores `UDI` y `Product ID` se excluyen del análisis porque no contienen información predictiva.

In [ ]:
# Grupos de columnas que se reutilizan en el resto del notebook
TARGET = "Machine failure"
FAILURE_MODES = ["TWF", "HDF", "PWF", "OSF", "RNF"]
NUMERIC_FEATURES = ["Air temperature [K]", "Process temperature [K]",
                    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
]
CATEGORICAL_FEATURES = ["Type"]
TEXT_COLUMNS = ["Product ID", "Type"]

### 2.1 Estadística descriptiva

`describe()` resume cada *feature* numérica: conteo, media, desviación estándar, mínimo, cuartiles y máximo. Se agregan dos medidas de forma:
- **Asimetría (*skewness*):** un valor cercano a 0 indica una distribución simétrica; valores mayores a 1 indican una cola larga hacia la derecha.
- **Curtosis (*kurtosis*):** mide qué tan pesadas son las colas. pandas calcula la curtosis en exceso, donde una distribución normal vale 0: valores positivos indican colas pesadas (más valores extremos de lo normal) y valores negativos indican colas ligeras o una distribución más plana.

In [ ]:
# Resumen estadístico de las features numéricas (una fila por feature)
summary = df[NUMERIC_FEATURES].describe().T
summary["skewness"] = df[NUMERIC_FEATURES].skew()
summary["kurtosis"] = df[NUMERIC_FEATURES].kurt()  # curtosis en exceso (normal = 0)
summary.round(4)

Para las columnas de texto, `describe()` muestra el número de valores distintos (`unique`), el valor más frecuente (`top`) y su frecuencia (`freq`).

In [ ]:
# Resumen de las columnas de texto: valores únicos, valor más frecuente y su frecuencia
df[TEXT_COLUMNS].describe().T

In [ ]:
# Conteo y porcentaje por valor de cada variable categórica
# (Product ID se omite: sus 10,000 valores son únicos y la lista no aporta información)
for column in CATEGORICAL_FEATURES:
    counts = pd.concat([df[column].value_counts(),
                        df[column].value_counts(normalize=True).mul(100).round(2)],
                       axis=1, keys=["count", "percent"],
    )
    print(counts)
    print("-" * 45)

**Observación:** según la documentación, las variantes de calidad se reparten como L 50%, M 30% y H 20%. En los datos, M coincide (29.97%), pero L representa el 60% y H solo el 10.03%. La diferencia no afecta el análisis, pero se documenta porque los datos no reflejan exactamente la descripción de su fuente.

### 2.2 Balance de clases del *target*

En un problema de clasificación es indispensable revisar qué proporción de registros pertenece a cada clase. Si una clase es muy minoritaria, la métrica *accuracy* puede ser engañosa: un modelo que siempre predice "sin falla" acertaría casi siempre sin detectar ninguna falla.

In [ ]:
# Conteo y porcentaje de cada clase del target
target_counts = pd.DataFrame({"count": df[TARGET].value_counts(),
                              "percent": df[TARGET].value_counts(normalize=True).mul(100).round(2),
})
print(target_counts)

# Gráfica de barras del balance de clases
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Sin falla (0)", "Falla (1)"], target_counts["count"].sort_index(),
       color=[TARGET_COLORS[0], TARGET_COLORS[1]],
)
for i, value in enumerate(target_counts["count"].sort_index()):
    ax.text(i, value, f"{value} ({value / len(df):.2%})", ha="center", va="bottom")
ax.set_title("Balance de clases: Machine failure")
ax.set_ylabel("Número de registros")
plt.tight_layout()
plt.show()

### 2.3 Modos de falla y consistencia de etiquetas

Según la documentación, `Machine failure` vale 1 cuando ocurre al menos uno de los cinco modos de falla. Se verifica si esa regla se cumple en los datos, comparando el *target* contra la presencia de algún modo de falla.

In [ ]:
# Número de registros con cada modo de falla
print("Registros por modo de falla:")
print(df[FAILURE_MODES].sum().to_string())

# Número de modos de falla activos en cada registro
n_modes = df[FAILURE_MODES].sum(axis=1)

# Tabla cruzada: target contra "tiene al menos un modo de falla"
print("\nTarget vs. presencia de algún modo de falla:")
print(pd.crosstab(df[TARGET], n_modes > 0,
                  rownames=[TARGET], colnames=["Algún modo de falla"],
))

print("\nFallas sin ningún modo registrado    :", ((df[TARGET] == 1) & (n_modes == 0)).sum())
print("Modo de falla sin falla registrada   :", ((df[TARGET] == 0) & (n_modes > 0)).sum())
print("Registros con más de un modo de falla:", (n_modes > 1).sum())

### 2.4 Distribución de las *features* numéricas

Los histogramas muestran la forma de cada distribución: simetría, sesgo, posibles valores atípicos y rangos.

In [ ]:
# Un histograma por feature numérica
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.flat, NUMERIC_FEATURES):
    sns.histplot(data=df, x=feature, bins=40, color=TARGET_COLORS[0], ax=ax)
    ax.set_title(feature)
    ax.set_ylabel("Frecuencia")
axes.flat[-1].set_visible(False)  # la cuadrícula tiene 6 espacios y solo hay 5 features
fig.suptitle("Distribución de las features numéricas", fontsize=14)
plt.tight_layout()
plt.show()

### 2.5 Relación de las *features* con el *target*

Los diagramas de caja (*boxplots*) comparan la distribución de cada *feature* entre registros sin falla y con falla. Si las cajas se separan, la *feature* aporta información para distinguir las clases.

In [ ]:
# Boxplot de cada feature numérica, separado por clase del target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.flat, NUMERIC_FEATURES):
    sns.boxplot(data=df, x=TARGET, y=feature, hue=TARGET,
                palette=TARGET_COLORS, legend=False, ax=ax,
    )
    ax.set_title(feature)
    ax.set_xticks([0, 1], ["Sin falla (0)", "Falla (1)"])
    ax.set_xlabel("")
axes.flat[-1].set_visible(False)
fig.suptitle("Features numéricas por clase del target", fontsize=14)
plt.tight_layout()
plt.show()

# Mediana de cada feature por clase, como apoyo numérico a la gráfica
df.groupby(TARGET)[NUMERIC_FEATURES].median().round(1)

In [ ]:
# Tasa de falla (%) por tipo de producto
failure_rate_by_type = (df.groupby("Type")[TARGET].mean()
                        .mul(100)
                        .reindex(["L", "M", "H"])
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(failure_rate_by_type.index, failure_rate_by_type.values, color=TARGET_COLORS[1])
for i, value in enumerate(failure_rate_by_type.values):
    ax.text(i, value, f"{value:.2f}%", ha="center", va="bottom")
ax.set_title("Tasa de falla por Type")
ax.set_xlabel("Type (calidad del producto)")
ax.set_ylabel("Tasa de falla (%)")
plt.tight_layout()
plt.show()

### 2.6 Correlación entre variables

La matriz de correlación de Pearson mide relaciones **lineales** entre pares de variables, de -1 (relación inversa perfecta) a 1 (relación directa perfecta). Sirve para detectar *features* redundantes entre sí y relaciones lineales con el *target*. Una correlación baja no descarta relaciones no lineales.

In [ ]:
# Matriz de correlación de las features numéricas y el target
corr = df[NUMERIC_FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=ax,
)
ax.set_title("Matriz de correlación (Pearson)")
plt.tight_layout()
plt.show()

### 2.7 Hallazgos del EDA

1. **Fuerte desbalance de clases:** solo el 3.39% de los registros (339 de 10,000) son fallas. *Accuracy* no es una métrica adecuada para este problema; en el Paso 5 se usarán métricas centradas en la clase minoritaria (*recall*, *precision*, F1). La partición del Paso 4 debe ser estratificada para conservar esta proporción en entrenamiento y prueba.
2. **Etiquetas inconsistentes:** la regla documentada ("falla = al menos un modo de falla") no se cumple en 27 registros: 9 fallas sin ningún modo registrado y 18 registros con modo de falla (`RNF`) pero sin falla. Además, 24 registros presentan más de un modo de falla a la vez, lo cual es válido. Las inconsistencias se tratan en el Paso 3.
3. **Distribuciones:** las temperaturas y el torque tienen distribuciones aproximadamente simétricas, y el torque es prácticamente normal (*skewness* -0.0095, *kurtosis* -0.0132); el desgaste de la herramienta (`Tool wear`) es casi uniforme entre 0 y ~200 min, lo que confirma su curtosis negativa (-1.1667); `Rotational speed` tiene una cola larga hacia la derecha (*skewness* 1.9932) y colas pesadas (*kurtosis* 7.3929). Sus valores altos parecen atípicos, pero se revisan en el Paso 3 antes de decidir, porque pueden ser físicamente válidos.
4. **Relación con el *target*:** las fallas se asocian con mayor torque (mediana 53.7 vs. 39.9 Nm), mayor desgaste de herramienta (165 vs. 107 min), menor velocidad de rotación (1,365 vs. 1,507 rpm) y temperatura del aire ligeramente mayor. En torque y velocidad, las fallas aparecen en **ambos extremos** (muy alta velocidad con torque muy bajo, y viceversa), un patrón coherente con la falla de potencia (`PWF`).
5. **Type:** la tasa de falla disminuye con la calidad del producto: L 3.92%, M 2.77%, H 2.09%. `Type` aporta información y se conservará como *feature*. Su distribución (L 60%, M 30%, H 10%) difiere de la documentada (50/30/20).
6. **Correlación:** hay dos pares de *features* fuertemente correlacionados: temperatura del aire con temperatura del proceso (0.88) y velocidad de rotación con torque (-0.88), lo que es consistente con cómo se generó el dataset. Ninguna *feature* tiene una correlación lineal alta con el *target* (máximo 0.19, torque); dado el patrón de extremos del punto 4, la relación con las fallas es no lineal, lo que se considerará al elegir el modelo *baseline*.